# 03 — Stage 1: YOLOv8 Detector (SageMaker version)

SageMaker port of `03_stage1_yolo.ipynb` (which is Colab-only). Runs on **ml.g5.xlarge (A10G 24GB)** → `batch=16 imgsz=1024` fits (the T4 16GB OOM'd at epoch ~24).

Paths are on EBS (`~/SageMaker/...`, persist across Stop/Start). DENTEX must already be at `~/SageMaker/opg-data/data/dentex/` (see `00_setup_sagemaker.ipynb` Cell 4 — gdown/S3).

## Cell 1 — Paths (EBS) + pull repo + GPU

In [ ]:
import os, subprocess, torch
DATA_ROOT = os.path.expanduser('~/SageMaker/opg-data')   # data + checkpoints (EBS)
REPO      = os.path.expanduser('~/SageMaker/opg-live')   # code
YOLO_DIR  = os.path.expanduser('~/SageMaker/yolo')       # converted YOLO dataset (scratch)
RUNS      = os.path.expanduser('~/SageMaker/runs')       # training runs

if not os.path.exists(REPO):
    subprocess.run(['git','clone','https://github.com/AndreasTopuh/opg-live.git',REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'fetch','origin'], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','origin/main'], check=True)

print('GPU      :', torch.cuda.get_device_name(0))
print('DATA_ROOT:', DATA_ROOT)

## Cell 2 — Install Ultralytics

In [ ]:
%pip install -q ultralytics
from ultralytics import YOLO
print('ultralytics ready')

## Cell 3 — Check DENTEX is on EBS

In [ ]:
import glob
js = glob.glob(f'{DATA_ROOT}/data/dentex/**/*disease*.json', recursive=True)
if js:
    print('\u2705 DENTEX found:', js[0])
else:
    print('\u26a0\ufe0f DENTEX NOT found under', f'{DATA_ROOT}/data/dentex/')
    print('   Upload it first (gdown from Drive / S3) \u2014 see 00_setup_sagemaker.ipynb Cell 4.')

## Cell 4 — Convert DENTEX \u2192 YOLO (stratified split + background + clamp)

In [ ]:
!python {REPO}/scripts/stage1/dentex_to_yolo.py --drive {DATA_ROOT} --out {YOLO_DIR}
!echo '--- contoh label ---'; head -3 $(ls {YOLO_DIR}/labels/train/*.txt | head -1)

## Cell 5 — Train YOLOv8m
A10G 24GB \u2192 `batch=16 imgsz=1024` fits. If OOM, drop to `batch=8`.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8m.pt')
results = model.train(
    data=f'{YOLO_DIR}/dentex.yaml',
    epochs=80, imgsz=1024, batch=16,
    project=RUNS, name='dentex', exist_ok=True,
    patience=20,
)

## Cell 5.b — Training curves (run anytime; results.csv updates each epoch)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
RUN = f'{RUNS}/dentex'
df = pd.read_csv(f'{RUN}/results.csv'); df.columns = [c.strip() for c in df.columns]
best = df.loc[df['metrics/mAP50-95(B)'].idxmax()]
print(f"Epoch done: {len(df)} | best epoch {int(best['epoch'])} | "
      f"mAP50 {best['metrics/mAP50(B)']:.4f} | mAP50-95 {best['metrics/mAP50-95(B)']:.4f}")
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].plot(df['epoch'], df['train/box_loss'], label='box'); ax[0].plot(df['epoch'], df['train/cls_loss'], label='cls'); ax[0].plot(df['epoch'], df['train/dfl_loss'], label='dfl')
ax[0].set_title('train losses'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50'); ax[1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95')
ax[1].set_title('val mAP'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.3)
ax[2].plot(df['epoch'], df['metrics/precision(B)'], label='precision'); ax[2].plot(df['epoch'], df['metrics/recall(B)'], label='recall')
ax[2].set_title('val P / R'); ax[2].set_xlabel('epoch'); ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Cell 6 — Save best.pt to EBS (+ optional S3 backup)

In [ ]:
import shutil, os
os.makedirs(f'{DATA_ROOT}/checkpoints', exist_ok=True)
src = f'{RUNS}/dentex/weights/best.pt'
dst = f'{DATA_ROOT}/checkpoints/yolov8_dentex.pt'
shutil.copy(src, dst)
print('\u2705 saved', dst)
# optional durable backup:
# !aws s3 cp {dst} s3://opg-live-235665523776/checkpoints/yolov8_dentex.pt

## Cell 7 — Eval: mAP global + per-class

In [ ]:
from ultralytics import YOLO
model = YOLO(f'{DATA_ROOT}/checkpoints/yolov8_dentex.pt')
m = model.val(data=f'{YOLO_DIR}/dentex.yaml', imgsz=1024)
print('mAP50    :', round(float(m.box.map50), 4))
print('mAP50-95 :', round(float(m.box.map), 4))
for i, ap in enumerate(m.box.maps):
    print(f'  {m.names[i]:20s} mAP50-95 {ap:.4f}')

## Cell 8 — Preview detection on 1 val image

In [ ]:
import glob, matplotlib.pyplot as plt, cv2
img = sorted(glob.glob(f'{YOLO_DIR}/images/val/*.png'))[0]
res = model.predict(img, imgsz=1024, conf=0.25)[0]
plt.figure(figsize=(16, 7)); plt.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
for b in res.boxes:
    print(res.names[int(b.cls)], 'conf', round(float(b.conf), 3))

## \u2705 Deliverable
- [ ] `yolov8_dentex.pt` in `~/SageMaker/opg-data/checkpoints/`
- [ ] mAP50 / mAP50-95 global + per-class logged
- [ ] training curves + preview look reasonable

\u26a0\ufe0f **STOP the instance** when done (credit burns while running).